# AI 챌린지 반영 · Level 3 단계별 연습 — 버전 3

**로컬 CPU용 창작 연습자료입니다. 실제 AI 챌린지 사진이 아닙니다.** 이전에 생성된 합성 도형 510장(품질 검사 후 508장)과 test 160장, 별도 합성 도형으로 사전학습한 작은 CNN을 재사용합니다. 인터넷·Colab·Qwen·양자화·LoRA 설치는 필요 없습니다.

기존 `(260908)_baseline_colab.ipynb`의 ‘데이터 → 모델 → Dataset → 학습/검증 → 추론 → CSV’ 흐름과 코드 제공 방식을 참고했습니다. 실제 Level 3 모델·제출 형식·통과 점수는 아직 미확정이며 CSV는 이 연습의 가정입니다.

**버전 3: 직접구현**. 세 버전은 같은 데이터·함수·최종 정답을 공유합니다. V1 13개의 짧은 빈칸 → V2 9개의 짧은 블록 → V3 주요 함수 구현 순으로 공부하세요. **이번에는 V1만 먼저 시작하세요.**

## 실행 지도
1. 공통 준비와 설정 셀은 실행만 합니다.
2. 각 문항에서 `TODO`만 채우고 아래의 ‘실행만 하세요’ 셀을 실행합니다.
3. 실행이 안 되면 현재 문항의 빈칸과 오류 메시지를 확인합니다. Q1 미완성도 Q2~Q4의 준비 자료를 막지 않습니다.
4. Q3·Q4가 끝나면 이 버전의 출력 폴더에서 CSV와 최고 가중치를 확인합니다.
5. 최초 목표는 점수 경쟁보다 **흐름을 이해하고 정상 CSV를 만드는 것**입니다. 이후 선택 실험으로 개선합니다.

데이터: 원/사각형/삼각형/십자 합성 RGB. CSV의 `path`는 PNG 경로이며 실제 챌린지의 이미지가 아닙니다. 제공 설정으로도 높은 점수가 보장되지는 않습니다.

In [ ]:
VERSION='V3'
# 실행만 하세요: 필요한 도구와 데이터 로드
from pathlib import Path
import sys, copy, json, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, confusion_matrix, mean_squared_error, cohen_kappa_score
ROOT=next((p for p in [Path.cwd(),Path.cwd().parent] if (p/'practice_support.py').is_file()),None)
if ROOT is None: raise FileNotFoundError('문제 폴더 또는 정답_채점용 폴더에서 여세요')
sys.path.insert(0,str(ROOT.resolve()))
from practice_support import (CLASSES,load_data,ready_splits,test_images,load_q4,load_backbone,ready_loaders,save_submission as support_save_submission,validate_json_schema)
torch.manual_seed(42);np.random.seed(42);random.seed(42);torch.set_num_threads(2)
meta,train_images=load_data()
print('메타데이터:',meta.shape,'이미지 배열:',train_images.shape,'클래스:',CLASSES)

## 설정 — 먼저 그대로 실행
빈칸을 완성한 뒤 이 숫자들을 하나씩 바꿔봅니다.

In [ ]:
# [제공 코드] 이 셀에서는 성능 개선을 위한 숫자만 바꿉니다. 빈칸이 아닙니다.
# 먼저 기본값으로 CSV가 생성되는지 확인하고, 그 뒤 하나씩 바꾸세요.
CLS_LR = 0.001             # 분류 학습률
CLS_EPOCHS = 10            # 분류 학습 횟수
CLS_UNFREEZE_LAST = True   # 마지막 블록도 학습할지
REG_LR = 0.003             # 회귀 학습률
REG_EPOCHS = 3             # 각 회귀 설정의 학습 횟수
OUTPUT_DIR = ROOT/'제출물'/VERSION
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)

def save_submission(frame,sample_file,name):
    # 버전별 출력 폴더를 사용해 다른 버전의 결과를 덮어쓰지 않습니다.
    return support_save_submission(frame,sample_file,name,folder=OUTPUT_DIR)
print('이 버전의 출력 폴더:',OUTPUT_DIR)

## 1. 이미지 품질 점검과 데이터 나누기
제공된 CSV를 읽고 잘못된 레이블·인덱스·검은 이미지·중복 ID를 제외합니다. 정상/검은 이미지를 눈으로 비교하고, 남은 자료를 클래스 비율을 고려하여 train/valid 8:2로 나눕니다.

**이미 제공:** 데이터 로딩, 검사 순서, 그림 표시, 분할 확인. `meta`는 표, `train_images`는 이미지 배열입니다.

함수의 입출력과 Reference를 보고 구현하세요.

In [ ]:
# 작성하세요
def inspect_and_split(meta, images):
    # 반환: 중복/손상 제거 후 split 열(train 또는 valid)을 가진 DataFrame
    raise NotImplementedError('Q1을 완성하세요')

answer_q1 = ''  # 데이터 품질과 불균형 관찰


In [ ]:
# 실행만 하세요. 오류가 나면 현재 문항의 TODO를 확인하세요.
try:
    # 실행만 하세요
    q1_frame=inspect_and_split(meta,train_images)
    assert q1_frame['id'].is_unique and set(q1_frame['split'])=={'train','valid'}
    print('정제 후 행 수:',len(q1_frame),'분할:',q1_frame['split'].value_counts().to_dict())
except Exception as exc:
    print('Q1 확인:',type(exc).__name__,str(exc))

## 2. 이미지 변환·증강과 불균형 처리
CSV의 path로 PNG를 읽어 `(3,H,W)`의 0~1 실수 텐서로 바꿉니다. 학습에서만 좌우 반전하고, 희소 클래스가 더 자주 뽑히도록 역빈도 샘플링을 사용합니다.

**이미 제공:** Image.open, 학습/검증 구분과 Dataset의 기본 구조. Q1을 못 풀어도 `ready_splits()`로 진행합니다.

함수의 입출력과 Reference를 보고 구현하세요.

In [ ]:
# 작성하세요
class PracticeDataset(Dataset):
    def __init__(self, rows, images, training):
        raise NotImplementedError('Q2 Dataset을 작성하세요')
    def __len__(self):
        raise NotImplementedError()
    def __getitem__(self, index):
        raise NotImplementedError()

def build_weighted_loader(rows, images, batch_size=32):
    # 반환: train DataLoader; 역빈도 샘플러 사용
    raise NotImplementedError('Q2 샘플러를 작성하세요')
answer_q2 = ''


In [ ]:
# 실행만 하세요. 오류가 나면 현재 문항의 TODO를 확인하세요.
try:
    # 실행만 하세요
    ready=ready_splits();q2_loader=build_weighted_loader(ready,train_images)
    xb,yb=next(iter(q2_loader));print('입력:',tuple(xb.shape),xb.dtype,'범위:',float(xb.min()),float(xb.max()),'정답:',tuple(yb.shape),yb.dtype)
    assert xb.ndim==4 and xb.shape[1:]==(3,32,32) and yb.dtype==torch.long
except Exception as exc:
    print('Q2 확인:',type(exc).__name__,str(exc))

## 3. 제공 모델로 분류 학습하고 CSV 만들기
제공된 소형 사전학습 백본을 재사용합니다. 앞 블록은 고정하고 마지막 블록과 새 출력층을 학습합니다. 최고 validation macro-F1 가중치를 복원한 모델로 `q3_submission.csv`를 만듭니다.

**이미 제공:** 백본 가중치, DataLoader, 입출력 규칙, 지표, 저장 및 ID 정렬 코드. logits는 `(B,4)`, 정답은 long `(B,)`입니다.

함수의 입출력과 Reference를 보고 구현하세요.

In [ ]:
# 작성하세요
def make_classifier():
    # 반환: 학습 범위가 설정된 nn.Module
    raise NotImplementedError('Q3 모델')

def train_classifier(model, train_loader, valid_loader, epochs=10):
    # 반환: (최선 가중치로 복원한 model, 검증 macro-F1)
    raise NotImplementedError('Q3 학습')

def classify_test(model):
    # 반환: id,label DataFrame; 샘플 순서
    raise NotImplementedError('Q3 추론')
answer_q3 = ''


In [ ]:
# 실행만 하세요. 오류가 나면 현재 문항의 TODO를 확인하세요.
try:
    # 실행만 하세요
    torch.manual_seed(42);random.seed(42)
    q3_train,q3_valid=ready_loaders(train_images,ready_splits())
    q3_model=make_classifier();q3_model,q3_f1=train_classifier(q3_model,q3_train,q3_valid,epochs=CLS_EPOCHS)
    q3_submission=classify_test(q3_model)
    save_submission(q3_submission,'q3_sample_submission.csv','q3_submission.csv')
    print('Q3 최고 검증 macro-F1:',q3_f1)
except Exception as exc:
    print('Q3 확인:',type(exc).__name__,str(exc))

## 4. 이미지 회귀와 최고 모델 선택
같은 이미지로 창작된 0~100 연속 점수를 예측합니다. loss는 score/100에 대한 MSE, 비교 지표는 원래 단위의 RMSE입니다. head만 학습한 경우와 마지막 블록도 학습한 경우를 비교하고 `q4_submission.csv`를 만듭니다.

**이미 제공:** 두 실험을 실행하는 코드와 등급 QWK 계산·ID 정렬 코드. RMSE는 낮을수록, QWK는 높을수록 좋습니다.

함수의 입출력과 Reference를 보고 구현하세요.

In [ ]:
# 작성하세요
def make_regressor(unfreeze_last):
    raise NotImplementedError('Q4 모델')
def train_regressor(model, train_loader, valid_loader, thresholds, epochs=3):
    # 반환: (최선 모델, 최고 검증 RMSE, 그때의 QWK)
    raise NotImplementedError('Q4 학습')
def predict_regression(model):
    # 반환: id,score DataFrame; 샘플 순서
    raise NotImplementedError('Q4 추론')
answer_q4 = ''


In [ ]:
# 실행만 하세요. 오류가 나면 현재 문항의 TODO를 확인하세요.
try:
    # 실행만 하세요: 같은 train/valid에서 2가지 학습 범위를 비교
    q4_rows,q4_thresholds=load_q4()
    q4_results=[]
    for unfreeze in [False,True]:
        # 매 설정마다 동일한 난수로 모델과 로더를 준비합니다.
        torch.manual_seed(42);random.seed(42)
        tr,va=ready_loaders(train_images,q4_rows,regression=True)
        model=make_regressor(unfreeze)
        model,score,qwk=train_regressor(model,tr,va,q4_thresholds,epochs=REG_EPOCHS)
        q4_results.append((score,qwk,unfreeze,model))
    q4_best=min(q4_results,key=lambda item:item[0])
    print('Q4 선택: 마지막 블록 학습=',q4_best[2],'RMSE=',q4_best[0],'QWK=',q4_best[1])
    torch.save(q4_best[3].state_dict(),OUTPUT_DIR/'q4_best.pt')
    q4_submission=predict_regression(q4_best[3])
    save_submission(q4_submission,'q4_sample_submission.csv','q4_submission.csv')
except Exception as exc:
    print('Q4 확인:',type(exc).__name__,str(exc))

## 5. 미확정 정보와 JSON을 다루는 프롬프트
합성 고객 문의에서 모델명·문제·배터리 시간·보증·수리 여부를 JSON으로 추출하는 프롬프트입니다. 없거나 모르는 정보는 null, 명시적 부정은 false, 한 시간 반은 90분입니다.

**이미 제공:** 키 목록·출력 자료형·검사기·수작업 출력 예시. 실제 LLM은 제공하거나 실행하지 않습니다. 프롬프트와 수작업 예상 JSON을 비교합니다.

함수의 입출력과 Reference를 보고 구현하세요.

In [ ]:
# 작성하세요
# format(note_id=..., note=...)이 가능하도록 JSON 예시의 중괄호는 {{ }}로 이스케이프하세요.
prompt_template = """
여기에 프롬프트를 작성하세요.
입력 ID: {note_id}
문의: {note}
"""
manual_examples = []  # 수작업으로 작성한 JSON 객체 2개 이상


In [ ]:
# 실행만 하세요. 오류가 나면 현재 문항의 TODO를 확인하세요.
try:
    assert '__V' not in prompt_template, '프롬프트의 빈칸을 먼저 채우세요.'
    # 실행만 하세요: 스키마는 확인하지만 추출 내용의 사실성은 직접 검토해야 합니다.
    notes=json.loads((ROOT/'data/q5_notes.json').read_text(encoding='utf8'))
    print(prompt_template.format(note_id=notes[0]['id'],note=notes[0]['note']))
    assert len(manual_examples)>=2
    for example in manual_examples:
        assert validate_json_schema(example),f'키·자료형을 확인하세요: {example}'
        print(json.dumps(example,ensure_ascii=False))
    (OUTPUT_DIR/'q5_prompt.txt').write_text(prompt_template,encoding='utf8')
    print('프롬프트 저장 완료. 실제 LLM 응답은 생성하지 않았습니다.')
except Exception as exc:
    print('Q5 확인:',type(exc).__name__,str(exc))

## 선택: 숫자 하나를 바꿔 성능 비교
Q3 기본 모델을 만들었을 때만 사용합니다. validation이 좋아진 경우만 최종 CSV와 가중치를 교체합니다.

In [ ]:
# [선택 실험 / 제공 코드] 기본 CSV 생성 후에만 실행합니다.
# 첫 연습에서는 False로 두세요. 나중에 True로 바꾸고 아래 설정 하나만 변경합니다.
RUN_EXTRA_TRIAL = False
TRY_LR = 0.0003
TRY_EPOCHS = 10
TRY_UNFREEZE_LAST = True

if RUN_EXTRA_TRIAL:
    if 'q3_model' not in globals():
        raise RuntimeError('Q3 기본 학습을 먼저 완료하세요.')
    # 현재 최고 모델은 다른 객체로 보관하므로 후보 실험이 악화되어도 잃지 않습니다.
    previous_model,previous_score=q3_model,q3_f1
    old_lr,old_flag=CLS_LR,CLS_UNFREEZE_LAST
    CLS_LR,CLS_UNFREEZE_LAST=TRY_LR,TRY_UNFREEZE_LAST
    try:
        torch.manual_seed(42);random.seed(42)
        tr,va=ready_loaders(train_images,ready_splits())
        candidate=make_classifier()
        candidate,candidate_score=train_classifier(candidate,tr,va,epochs=TRY_EPOCHS)
    finally:
        CLS_LR,CLS_UNFREEZE_LAST=old_lr,old_flag
    print('기존 valid macro-F1:',previous_score,'후보:',candidate_score)
    if candidate_score>previous_score:
        q3_model,q3_f1=candidate,candidate_score
        print('검증이 개선되어 후보를 선택합니다.')
    else:
        q3_model,q3_f1=previous_model,previous_score
        print('기존 모델을 유지합니다.')
    # 학습 함수가 임시 후보 파일을 썼더라도 최종 선택 가중치로 다시 저장합니다.
    torch.save(q3_model.state_dict(),OUTPUT_DIR/'q3_best.pt')
    save_submission(classify_test(q3_model),'q3_sample_submission.csv','q3_submission.csv')
else:
    print('추가 학습 생략. 먼저 기본 문제를 완성한 뒤 숫자 하나를 바꾸어 비교하세요.')

## 끝 확인
CSV 두 개와 q5_prompt.txt를 열어보세요. ID·열·행 수·값 범위를 확인합니다. 실제 시험 제출 규칙과 코드 리뷰 요구는 현장 지문을 따릅니다.